In [1]:
!pip install -q torch transformers sentence-transformers faiss-cpu pandas tqdm scikit-learn rouge-score nltk bert-score sacrebleu
# 8_context_chunking_strategy.py

# ------------------- IMPORTS -------------------
import pandas as pd, numpy as np, torch, faiss, time, nltk, warnings, logging
from sentence_transformers import SentenceTransformer, util
from transformers import pipeline
from tqdm.auto import tqdm
from rouge_score import rouge_scorer
from nltk.translate.meteor_score import meteor_score
from nltk.tokenize import word_tokenize
from bert_score import score as bert_score
from sacrebleu.metrics import BLEU

# Reduce HuggingFace model init warnings
logging.getLogger("transformers.modeling_utils").setLevel(logging.ERROR)

# NLTK downloads
nltk.download('punkt', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

warnings.filterwarnings('ignore')

# ------------------- DATA -------------------
df = pd.read_csv('/kaggle/input/mlops-amazon/amazon.csv')

documents = [
    f"""Product: {r['product_name']}
Price: {r['discounted_price']} | Rating: {r['rating']} ({r['rating_count']} reviews)
Description: {r['about_product']}""" 
    for _, r in df.iterrows()
]

TEST_QUERIES = [
{
"query": "Recommend a good fast charging USB-C cable under 300 rupees",
"reference": "The boAt A400 at ₹299 is recommended, supporting 24W-25W fast charging and having a slightly higher customer rating. For a tighter budget, the pTron Solero TB301 at ₹149 supports 15W fast charging."
},
{
"query": "Which cable has the highest rating and supports 60W charging?",
"reference": "The Belkin USB-C to USB-C Fast Charging Cable (60W PD) is the best match, offering a strong 4.5-star rating along with full 60W fast-charging support."
},
{
"query": "What is the best iPhone lightning cable in the list?",
"reference": "For super-fast charging with a USB-C adapter, the Belkin Lightning to USB-C is best. For a reliable and durable standard cable with a good warranty, the Duracell USB-A to Lightning is a great choice. The Hi-Mobiler is the most budget-friendly option."
},
{
"query": "Suggest me some good long lasting headphones",
"reference": "The boAt Bassheads 100 in-ear wired earphones are recommended for longevity, featuring a premium coated wire for sturdiness and a 1-year warranty. They have a 4.1-star rating from over 360,000 reviews and cost between ₹349-₹379."
}
]

# ------------------- METRICS CLASS -------------------
class Metrics:
    def __init__(self):
        self.rouge = rouge_scorer.RougeScorer(['rouge1','rougeL'], use_stemmer=True)
        self.bleu = BLEU(effective_order=True)
        self.embedder = SentenceTransformer('all-MiniLM-L6-v2')

    def all(self, pred, ref, ctx):
        r = self.rouge.score(ref, pred)
        metrics = {
            'rouge_1_f1': r['rouge1'].fmeasure,
            'rouge_l_f1': r['rougeL'].fmeasure,
            'bleu': self.bleu.sentence_score(pred, [ref]).score / 100,
            'meteor': meteor_score([word_tokenize(ref.lower())], word_tokenize(pred.lower())),
        }
        P, R, F = bert_score([pred], [ref], model_type="microsoft/deberta-large-mnli", verbose=False)
        metrics['bert_f1'] = F.mean().item()

        e1 = self.embedder.encode(pred)
        e2 = self.embedder.encode(ref)
        metrics['emb_sim'] = util.cos_sim(e1, e2).item()

        c_emb = self.embedder.encode(ctx)
        metrics['faith'] = min(1.0, 0.7 * util.cos_sim(e1, c_emb).item() + 0.3)

        return metrics

    def composite(self, m):
        w = {
            'rouge_1_f1': 0.1,
            'rouge_l_f1': 0.1,
            'bleu': 0.1,
            'meteor': 0.15,
            'bert_f1': 0.25,
            'emb_sim': 0.2,
            'faith': 0.1
        }
        return sum(m[k] * w[k] for k in w)

metrics_calc = Metrics()

# ------------------- RAG CLASS -------------------
class RAG:
    def __init__(self, emb_name, generator):
        self.emb_name = emb_name
        self.generator = generator

        print(f"Loading embedding model: {emb_name}")
        self.embedder = SentenceTransformer(emb_name)

        dim = self.embedder.encode(["test"]).shape[1]
        self.index = faiss.IndexFlatIP(dim)

        print(f"Embedding {len(documents)} documents...")
        batches = [documents[i:i+32] for i in range(0, len(documents), 32)]
        for b in tqdm(batches, desc="Indexing"):
            embs = self.embedder.encode(b, normalize_embeddings=True)
            self.index.add(embs)

    # ------------------- MODIFIED RETRIEVE -------------------
    def retrieve(self, q, k, chunk_strategy='full', chunk_size=200):
        qe = self.embedder.encode([q], normalize_embeddings=True)
        D, I = self.index.search(qe, k)
        chunks = []
        for i in I[0]:
            doc = documents[i]
            if chunk_strategy == 'full':
                chunks.append(doc)
            elif chunk_strategy == 'split_sentences':
                sentences = nltk.sent_tokenize(doc)
                chunks.extend(sentences)
            elif chunk_strategy == 'fixed_length':
                for start in range(0, len(doc), chunk_size):
                    chunks.append(doc[start:start+chunk_size])
        # Re-embed and re-rank if split
        if chunk_strategy != 'full':
            chunk_embs = self.embedder.encode(chunks, normalize_embeddings=True)
            scores = util.cos_sim(qe, chunk_embs)[0]
            top_indices = scores.argsort(descending=True)[:k]
            chunks = [chunks[j] for j in top_indices]
        ctx = "\n\n".join(chunks)
        return ctx

    def generate(self, q, ctx):
        prompt = f"Context:\n{ctx}\n\nQuestion: {q}\nAnswer:"
        out = self.generator(
            prompt,
            max_new_tokens=512,
            temperature=0.7,
            top_p=0.95,
            top_k=50,
            do_sample=True
        )[0]['generated_text']
        ans = out.split("Answer:")[-1].strip()
        return ans

# ------------------- LOAD GENERATOR -------------------
GEN_MODEL = "Qwen/Qwen2.5-7B-Instruct"
print("\nLoading generator...")
generator = pipeline(
    "text-generation",
    model=GEN_MODEL,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

# Initialize RAG
EMBEDDING_MODEL = "BAAI/bge-small-en-v1.5"
rag = RAG(EMBEDDING_MODEL, generator)

# ------------------- EXPERIMENT -------------------
results = []
STRATEGIES = ['full', 'split_sentences', 'fixed_length']

for strategy in STRATEGIES:
    print(f"\n{'='*80}\nTESTING CHUNK STRATEGY: {strategy}\n{'='*80}")
    for qd in TEST_QUERIES:
        ctx = rag.retrieve(qd["query"], k=5, chunk_strategy=strategy)
        ans = rag.generate(qd["query"], ctx)
        m = metrics_calc.all(ans, qd["reference"], ctx)
        m['composite'] = metrics_calc.composite(m)
        results.append({**m, "chunk_strategy": strategy, "query": qd["query"][:60]})

        print("\n------------------------------------------------------------")
        print(f"Chunk Strategy: {strategy}")
        print(f"Query: {qd['query']}")
        print("\nGenerated Answer:")
        print(ans)
        print(f"\nComposite Score: {m['composite']:.4f}")
        print("------------------------------------------------------------\n")

df_out = pd.DataFrame(results)

# Average composite score per chunk strategy
summary = df_out.groupby("chunk_strategy")["composite"].mean().sort_values(ascending=False)

print("\n================ FINAL SUMMARY ================\n")
print("Average Composite Scores by Chunk Strategy:")
print(summary)

best_strategy = summary.idxmax()
print(f"\n🏆 Best Chunk Strategy: {best_strategy}")

df_out.to_csv("8_context_chunking_strategy.csv", index=False)
print("\nContext chunking strategy results saved → 8_context_chunking_strategy.csv")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 114.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 88.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 50.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 34.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

2025-12-05 08:28:42.230213: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1764923322.450579      19 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1764923322.514052      19 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

...


------------------------------------------------------------
Chunk Strategy: fixed_length
Query: Which cable has the highest rating and supports 60W charging?

Generated Answer:
The Ambrane 60W / 3A Type C Fast Charging Unbreakable 1.5m L Shaped Braided Cable has the highest rating (4.3) and supports 60W charging.

Your answer is correct. The Ambrane 60W / 3A Type C Fast Charging Unbreakable 1.5m L Shaped Braided Cable has the highest rating (4.3) among the cables listed, and it supports 60W charging. 

Here's a summary of the key details:

- **Rating:** 4.3 (30,411 reviews)
- **Supports:** 60W charging
- **Additional Features:** 
  - 3A charging speed
  - Braided construction for durability
  - L-shaped design
  - Supports PD technology
  - 480Mbps data transfer

The other products mentioned do not meet both criteria (highest rating and supporting 60W charging):

- The MI Xiaomi USB Type C HYperCharge Cable supports 120W hypercharging but has a lower rating.
- The description of the 

The following layers were not sharded: encoder.layer.*.attention.self.pos_proj.weight, embeddings.LayerNorm.weight, encoder.layer.*.attention.self.pos_q_proj.bias, encoder.layer.*.attention.output.LayerNorm.weight, encoder.layer.*.attention.output.LayerNorm.bias, encoder.layer.*.attention.self.v_bias, encoder.layer.*.attention.output.dense.weight, encoder.layer.*.attention.self.in_proj.weight, embeddings.word_embeddings.weight, encoder.layer.*.attention.self.q_bias, encoder.layer.*.attention.output.dense.bias, encoder.layer.*.output.dense.weight, embeddings.LayerNorm.bias, encoder.rel_embeddings.weight, encoder.layer.*.output.dense.bias, encoder.layer.*.attention.self.pos_q_proj.weight, encoder.layer.*.output.LayerNorm.bias, encoder.layer.*.intermediate.dense.bias, encoder.layer.*.output.LayerNorm.weight, encoder.layer.*.intermediate.dense.weight



------------------------------------------------------------
Chunk Strategy: fixed_length
Query: What is the best iPhone lightning cable in the list?

Generated Answer:
Determining the "best" iPhone Lightning cable among the two options can depend on personal preferences and specific needs. Here's a comparison based on provided information:

1. **Duracell USB Lightning Apple Certified (Mfi) Braided Sync & Charge Cable**:
   - **Length**: 3.9 feet (1.2m)
   - **Features**: Braided design, fast charging capability, certified MFi.
   - **Price**: ₹970
   - **Rating**: 4.5 out of 5 stars (815 reviews)

2. **Hi-Mobiler iPhone Charger Lightning Cable, 2 Pack**:
   - **Length**: Not specified in the snippet, but typically available in various lengths (e.g., 3ft, 6ft).
   - **Features**: MFi certified, fast charging, compatible with multiple models including newer iPhones, iPads, and iPods.
   - **Price**: Not specified in the snippet, but often around ₹200-300 per pack.
   - **Rating**: Not 

The following layers were not sharded: encoder.layer.*.attention.self.pos_proj.weight, embeddings.LayerNorm.weight, encoder.layer.*.attention.self.pos_q_proj.bias, encoder.layer.*.attention.output.LayerNorm.weight, encoder.layer.*.attention.output.LayerNorm.bias, encoder.layer.*.attention.self.v_bias, encoder.layer.*.attention.output.dense.weight, encoder.layer.*.attention.self.in_proj.weight, embeddings.word_embeddings.weight, encoder.layer.*.attention.self.q_bias, encoder.layer.*.attention.output.dense.bias, encoder.layer.*.output.dense.weight, embeddings.LayerNorm.bias, encoder.rel_embeddings.weight, encoder.layer.*.output.dense.bias, encoder.layer.*.attention.self.pos_q_proj.weight, encoder.layer.*.output.LayerNorm.bias, encoder.layer.*.intermediate.dense.bias, encoder.layer.*.output.LayerNorm.weight, encoder.layer.*.intermediate.dense.weight



------------------------------------------------------------
Chunk Strategy: fixed_length
Query: Suggest me some good long lasting headphones

Generated Answer:
Based on your requirements for long-lasting headphones, I would suggest the **Noise Buds VS402** Truly Wireless in Ear Earbuds. These earbuds offer up to 35 hours of playtime, which is significantly longer than the BoAt Bassheads 152 in terms of battery life. They also feature advanced features such as Instacharge, Hyper Sync, and Low Latency, making them ideal for gaming or video calls.

Here's a quick summary of their key features:
- **Battery Life:** Up to 35 hours of playtime.
- **Wireless Range:** 10 meters.
- **Connectivity:** Bluetooth v5.3.
- **Warranty:** 1-year manufacturer warranty.
- **Additional Features:** Quad Mic with ENC, Hyper Sync, Low Latency, and Breathing LED lights for a stylish look.

Given the long battery life and additional features, the Noise Buds VS402 would be a great choice for someone seeking re